In [1]:
import os
import glob
import pandas as pd
import re
from typing import Optional, Tuple, Iterable, List
import matplotlib.pyplot as plt
import PIL.Image as Image
from tqdm import tqdm

In [2]:
HOLDOUT_SET_DIR = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_12_margin_15_bg_wb_alif/images/holdout"
TRAIN_SET_DIR = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_12_margin_15_bg_wb_alif/images/train"
VAL_SET_DIR = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_12_margin_15_bg_wb_alif/images/val"

In [3]:
glob_pattern = os.path.join(TRAIN_SET_DIR, "*.jpg")
image_paths = glob.glob(glob_pattern)
img_names = [os.path.basename(p) for p in image_paths]
print(len(img_names))

glob_pattern = os.path.join(VAL_SET_DIR, "*.jpg")
image_paths = glob.glob(glob_pattern)
img_names_val = [os.path.basename(p) for p in image_paths]
print(len(img_names_val))

img_names_all = img_names + img_names_val
print(f"Total images in train and val sets: {len(img_names_all)}")

18018
1996
Total images in train and val sets: 20014


In [4]:
# 1) pattern with H_cam_### and env after that
_RE_CAM = re.compile(
    r"""
    _H_cam_(?P<height>\d+) # height, e.g. 234
    _(?P<env>[A-Za-z][A-Za-z0-9_]*) # env, e.g. spacesbooth / menivimOS_1
    _
    """,
    re.VERBOSE
)


# 2) pattern with H### and env after that (no cam)
_RE_NO_CAM = re.compile(
    r"""
    _H(?P<height>\d+) # height, e.g. 260
    _(?P<env>[A-Za-z][A-Za-z0-9_]*)# env
    _
    """,
    re.VERBOSE
)


def extract_height_env(filename: str) -> Optional[Tuple[int, str]]:
    """
    Returns (height:int, env:str) or None if cannot parse.
    """
    base = os.path.basename(filename)


    m = _RE_CAM.search(base)
    if not m:
        m = _RE_NO_CAM.search(base)
    if not m:
        return None

    height = int(m.group("height"))
    env = m.group("env")
    return height, env


def extract_all(files: Iterable[str]) -> List[Tuple[int, str]]:
    out = []
    for f in files:
        t = extract_height_env(f)
        if t is not None:
            out.append(t)
    return out

In [5]:
extracted = extract_all(img_names)
env_h_df = pd.DataFrame(extracted, columns=["height", "env"])
env_h_df.head()

,height,env
0,248,ViaSDesk
1,229,Kitchen
2,229,Kitchen
3,290,rivulliskit
4,239,MenivimEnt


In [8]:
n = len(env_h_df["env"].unique())
print(f"Unique environments: {n}")

img_folder = "env_examples"
os.makedirs(img_folder, exist_ok=True)


for env_name in tqdm(env_h_df["env"].unique()):
    plt.figure(figsize=(8,6))
    subset = [i for i in img_names if f"_{env_name}_" in i]
    
    img_pth = os.path.join(TRAIN_SET_DIR, subset[0])
    if os.path.exists(img_pth) is False:
        img_pth = os.path.join(VAL_SET_DIR, subset[0])
    img = Image.open(img_pth)
    plt.imshow(img)
    plt.title(f"{env_name} ({len(subset)} images)")
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(img_folder, f"{env_name}.jpg"))
    plt.close();

Unique environments: 217


100%|██████████| 217/217 [00:27<00:00,  7.78it/s]


In [9]:
env_h_df.to_csv("train_val_set_env_height.csv", index=False)